# Demo 3 · Pinned versus newest

**Slides:** "“latest” is a moving target" (slide 9), "Pin it, log it, cache it" (slide 15), demo step 03.

The same small evaluation runs against a pinned Gemini model and against a *floating* choice: code that
always picks the newest Flash model available today, the way a `-latest` alias or a "just use the newest" config does.
Every response records which model version answered, and responses are cached so a rerun makes no API calls.

Needs a Gemini API key stored as a Colab secret named `GOOGLE_API_KEY` (key icon in the left sidebar).
Runtime: any. The free tier allows 5 requests per minute per model, so the first run is paced and takes
about 3 to 4 minutes. Reruns come from the cache and take a second. Run it before recording, not on stage.

In [ ]:
REPO_URL = "https://github.com/keila-moral/reproducible-ml-demo"
import os
if not os.path.exists("/content/repo"):
    !git clone -q {REPO_URL} /content/repo
%cd /content/repo
!git log --oneline -1

In [ ]:
!pip install -q -U google-genai

## 1 · Connect and choose a pinned model

List the available Flash models and copy a **stable, explicitly versioned** ID into `PINNED_MODEL`.

In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GOOGLE_API_KEY"))
for m in client.models.list():
    if "flash" in m.name:
        print(m.name, "|", m.version)

In [ ]:
import re

PINNED_MODEL = "gemini-3.5-flash"   # an explicit, versioned model: change it only in a pull request

def newest_flash() -> str:
    """The floating choice: whichever gemini-X.Y-flash is newest today."""
    names = [m.name.removeprefix("models/") for m in client.models.list()]
    flash = [n for n in names if re.fullmatch(r"gemini-\d+(\.\d+)?-flash", n)]
    return max(flash, key=lambda n: tuple(int(x) for x in re.findall(r"\d+", n)))

FLOATING_MODEL = newest_flash()
print("pinned:  ", PINNED_MODEL)
print("floating:", FLOATING_MODEL, "(newest Flash today; next month it may be another model)")

CONFIG = types.GenerateContentConfig(temperature=0, seed=1234)

## 2 · Ask with logging and a cache

The cache key is a hash of model, prompt and config. Changing any of them is a new request; repeating them is free and identical.

In [ ]:
import hashlib, json, pathlib, re, time
from google.genai import errors

MIN_SECONDS_BETWEEN_CALLS = 13   # free tier: 5 requests per minute per model
_last_call = 0.0

def generate(model, prompt):
    """Call the API, pacing requests and waiting out 429 rate-limit errors."""
    global _last_call
    for attempt in range(5):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.monotonic() - _last_call)
        if wait > 0:
            time.sleep(wait)
        _last_call = time.monotonic()
        try:
            return client.models.generate_content(model=model, contents=prompt, config=CONFIG)
        except errors.ClientError as e:
            if e.code != 429:
                raise
            m = re.search(r"retry in ([0-9.]+)s", str(e))
            delay = float(m.group(1)) + 1 if m else 30
            print(f"rate limited, waiting {delay:.0f} s ...")
            time.sleep(delay)
    raise RuntimeError("still rate limited after 5 attempts")


CACHE = pathlib.Path(".gemini_cache/responses.jsonl")
CACHE.parent.mkdir(exist_ok=True)
cache = {}
if CACHE.exists():
    for line in CACHE.read_text().splitlines():
        rec = json.loads(line)
        cache[rec["key"]] = rec

def ask(model: str, prompt: str) -> dict:
    config = CONFIG.model_dump(exclude_none=True)
    key = hashlib.sha256(json.dumps([model, prompt, config], sort_keys=True).encode()).hexdigest()
    if key in cache:
        return {**cache[key], "cached": True}
    response = generate(model, prompt)
    rec = {"key": key, "model": model, "model_version": response.model_version,
           "prompt": prompt, "config": config, "text": (response.text or "").strip()}
    cache[key] = rec
    with CACHE.open("a") as f:
        f.write(json.dumps(rec) + "\n")
    return {**rec, "cached": False}

## 3 · A tiny evaluation set

In [ ]:
EVAL = [
    ("The update fixed every crash I had reported.", "positive"),
    ("The battery now lasts half as long as before.", "negative"),
    ("The package arrived on Tuesday.", "neutral"),
    ("Support answered quickly but could not solve the problem.", "negative"),
    ("Setup took two minutes and everything just worked.", "positive"),
    ("The manual is available in three languages.", "neutral"),
    ("I would not buy this again.", "negative"),
    ("Honestly better than I expected for the price.", "positive"),
]
TEMPLATE = ("Classify the sentiment of this review. Answer with exactly one word: "
            "positive, negative or neutral.\n\nReview: {text}")

## 4 · Run against the pinned model and the floating one

In [ ]:
import pandas as pd

def evaluate(model):
    rows = []
    for text, expected in EVAL:
        r = ask(model, TEMPLATE.format(text=text))
        answer = r["text"].lower().strip(" .")
        rows.append({"review": text[:40], "expected": expected, "answer": answer,
                     "correct": answer == expected, "model_version": r["model_version"],
                     "cached": r["cached"]})
    return pd.DataFrame(rows)

pinned = evaluate(PINNED_MODEL)
floating = evaluate(FLOATING_MODEL)
print(f"pinned   {PINNED_MODEL}: accuracy {pinned.correct.mean():.2f}, served by {set(pinned.model_version)}")
print(f"floating {FLOATING_MODEL}: accuracy {floating.correct.mean():.2f}, served by {set(floating.model_version)}")
pd.concat({"pinned": pinned, "floating": floating}, axis=1)

## 5 · Rerun: everything comes from the cache

Same answers, zero API calls. This is what makes an LLM evaluation reproducible next month.

In [ ]:
again = evaluate(PINNED_MODEL)
print("all cached:", again.cached.all(), "| identical answers:", (again.answer == pinned.answer).all())

## 6 · Log the evaluation

In [ ]:
!pip install -q mlflow
import mlflow
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("devfest-gemini-eval")
for label, model, df in [("pinned", PINNED_MODEL, pinned), ("floating", FLOATING_MODEL, floating)]:
    with mlflow.start_run(run_name=label):
        mlflow.log_params({"model": model, "model_version": ",".join(sorted(set(df.model_version))),
                           "temperature": 0, "seed": 1234, "n_items": len(df)})
        mlflow.log_metric("accuracy", float(df.correct.mean()))
        mlflow.log_artifact(str(CACHE))
print("logged")

## What to say

The two models may give the same answers today. The point is the record: when the floating choice moves,
the logged `model_version` shows it, and the cache lets you rerun last month's evaluation exactly.
Upgrade the pinned ID deliberately, in a pull request, with this evaluation rerun.